On DAP:
1) If needed, unzip the project:
<br> ```unzip graph-ensembles.zip```

2) Since one can't install ipykernel in .venv, you need to install the folder as a package on ```(virtual-env)```
<br> To install graph-ensembles, go in the terminal (virtual-env -- `cmd + j`) and type

    * `pip install --editable . --config-settings editable_mode=compat`
    * From the project dir, ```pip install -e .; pip install -r requirements.txt; python3 -c "import graph_ensembles" ``` (updates packages when changed) ;
        * If don't work, try with ```pip install .``` (packages in src are freezed)
    * ```python3 -c "import graph_ensembles"``` to check it was correctly installed;
    * ```pip install -r requirements.txt```

Useful command on DAP:

* For this proj, ```vars``` and ```plots``` are saved in `./outputs`, but in the old version in
```/home/inghero/data/corealgos/rmilocco/outputs/datasets/ING-Directed```
* Delete folders on dap: ```mc rm --force --purge folder```
* To save new plots, delete previous ones with
```mc rm --force --purge dap/corealgos/rmilocco/outputs/datasets/ING-Directed```

On local:
* macOS: Zip outside the ``graph-ensembles`` folder: 
<br>```zip -rX graph-ensembles.zip graph-ensembles -x "graph-ensembles/src/graph_ensembles.egg-info/*" ".*" "*/.*" "*/__pycache__/*"```

In [1]:
# import image module
from IPython.display import Image

# get the image
Image(url="../im1.png", width = 500)

Claim: by reconstructing the unobserved, we may close the gap between the total Page-Rank and Page-Rank only inside the ING-clients

1) Split the nodes into ING (`vI`) and `ROW` (`vR`);

2) Find ``eI`` as the edges only between ``vI`` + drop the ``vI`` with no links (in case, a `vI` is connected only to ROW-nodes);

3) Accordingly, define the edges as `intra` (`eI`), `inter`, `row`; 

4) Calculate the Page-Rank of only the `intra` nodes and compare it with the full graph PR;
Now, we expect that the 2 PR are different. So,
5) Freeze the `eI` and fit $\delta$ parameter as 
    * $L_I \stackrel{!}{=} \langle L_I \rangle(\delta_I) := \sum_{i \in I, j \in I} p_{ij}(\delta_I) \rightarrow $ the remaining probabilities are `inter, row` $p_{ir} := 1 - \exp(-\delta_{I} s^{out}_i s^{in}_r)$ with which we can sample and obtain the average $ \langle PR \rangle$ over the enhanced-network (`eI` (deterministic) + `e-sampled`);

    * $L_U = L_I + L_{inter} \stackrel{!}{=} L_I + \sum_{i \in I, r \in R} (p_{ir}(\delta_{U}) + p_{ri}(\delta_{U})) \rightarrow $ do the same calculations;

    * $L_{tot} \stackrel{!}{=} \langle L_{tot} \rangle(\delta_I) := L_I + \sum_{i \in I, r \in R} (p_{ir}(\delta_{tot}) + p_{ri}(\delta_{tot})) + \sum_{r \in R, t \in R} (p_{rt} + p_{tr})(\delta_{tot}) \rightarrow $ same;

In [2]:
# auto-reload the packages at every run
%load_ext autoreload
%autoreload 2

#display all the results not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

try:
    corpkey = True if os.environ['DSBOX_USERNAME'] else None
    %pip install matplotlib pandas scipy scikit-learn numba
except:
    corpkey = None

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os, sys
from fast_pagerank import pagerank_power
import graph_ensembles as ge
from graph_ensembles import sparse as sp
import graph_ensembles.utils as utils
import graph_ensembles.dependencies as dep

dataset_name = "ING" #"recNET"
dataset_direction = "Directed"
id_code, cg_method, year = "grid_id", "random", 2022

In [3]:
# note that inside the kwargs there is a copy of the pdtrans
pdtrans, kwargs, total_levels = \
    ge.dataset_loader(dataset_name, dataset_direction = dataset_direction,
                    corpkey = corpkey, id_code = id_code, cg_method = cg_method,
                    year = year, max_n_entries = 0)


Reading from local source


Find the (real) distribution of naics, using this code

In [4]:
# def _naics_num_firms(df):
#     df.loc[:, "nroffirms"] = 1
#     naics_num_firms = df.groupby(by = "payer_naics_code").sum().loc[:, "nroffirms"].to_dict()
#     return naics_num_firms
# naics_num_firms = _naics_num_firms(pdtrans.copy())

# save a dictionary
# with open('naics_num_firms.pkl', 'wb') as f:
#     pickle.dump(naics_num_firms, f)

In [5]:
# # load it
# with open('naics_num_firms.pkl', 'rb') as f:
#     naics_num_firms = pickle.load(f)

Define all the vertex and edges

In [6]:
e = pdtrans.loc[:, [f'payer_{id_code}', f'beneficiary_{id_code}']]
unique_nodes_from = lambda df: pd.DataFrame(data = pd.unique(df.to_numpy().ravel('K')), columns = ["id"])
v = unique_nodes_from(e)
e = pdtrans.iloc[:, ::2] # payer, beneficiary, amount_euro
e.columns = ["src", "dst", "amount"] 

e.head()

,src,dst,amount
0,0,7242,1905.827400
1,0,48709,370.242096
2,0,49585,39.347341
3,0,55106,402.814695
4,0,60333,17764.579201


Create the full graph to be used as test

In [7]:
kwargs_graph = {'name' : 'ING', 'level' : 0, 'corpkey' : corpkey}
g = sp.graphs.DiGraph(v, e, **kwargs_graph)
# g.load_or_create_degrees()

KeyboardInterrupt: 

In [ ]:
pr_g = pagerank_power(g.adj, p=0.85, max_iter=100,
                    tol=1e-06, personalize=None, reverse=True)

Create a split of ``v`` into `vI` and `vR`

In [ ]:
# set the perc of intra nodes to be freezed
N = v.shape[0]
perc_intra_nodes = 0.7
num_intra_nodes = int(perc_intra_nodes * N)

# fixed a seed, extract num_intra_nodes indexes for the vI nodes 
seed = 0
np.random.seed(seed)
idx_intra_nodes = np.random.choice(N, size = num_intra_nodes, replace=False)
vI = v.iloc[idx_intra_nodes].sort_values(by = "id", ignore_index = True)

In [ ]:
# find idx of edges containing v and filter edges
idx_with_both_ = lambda v: e['src'].isin(v['id']) & e['dst'].isin(v['id'])
edge_idx = lambda idx: e.loc[idx].reset_index(drop = True)

# containing vI, vR
idx_eI = idx_with_both_(vI)

# select the edge ING
eI = edge_idx(idx_eI)

# re-obtain the vI nodes, since there may be an ING node connected only to ROW nodes
vI = unique_nodes_from(eI.iloc[:,:2])

# select the rest-of-the-world vertex, but including the ones discarded from vI
idx_v_row = np.setdiff1d(np.squeeze(v.values), np.squeeze(vI.values), assume_unique=True)
vR = pd.DataFrame(data = idx_v_row, columns = ["id"])

# select the edge ROW
eR = edge_idx(~idx_eI)

assert vI.shape[0] + vR.shape[0] == N, "Some nodes are not present either in the ING or ROW nodes"
assert eI.shape[0] + eR.shape[0] == e.shape[0], "Some edges are not present either in the ING or ROW nodes"

print(f'-Nodes splitted in vI.shape[0], vR.shape[0]: {vI.shape[0], vR.shape[0]}',)
print(f'-vI.shape[0] / v.shape[0]: {np.round(vI.shape[0] / v.shape[0], 3) * 100} %',)
print(f'-The target percentage was {perc_intra_nodes * 100} %')

# redefine the perc_intra_nodes
perc_intra_nodes = round(vI.shape[0] / v.shape[0], 3)

-Nodes splitted in vI.shape[0], vR.shape[0]: (75468, 44144)
-vI.shape[0] / v.shape[0]: 63.1 %
-The target percentage was 70.0 %


Create the `gI` and calculate its Page-Rank

In [ ]:
kwargs_graph.update({'perc_intra_nodes' : perc_intra_nodes, 'seed' : seed, 'graph_kind' : "intra",})
gI = sp.graphs.DiGraph(vI, eI, **kwargs_graph)
# g.load_or_create_degrees()

pr_gI = pagerank_power(gI.adj, p=0.85, max_iter=100,
                   tol=1e-06, personalize=None, reverse=True)

In [ ]:
# pr_g has all the nodes, therefore filter only the vI
# id_dict = {node : idx}. The map is gI_nodes --> has keys of id_dict_Full --> returning the idx_Full
idx_IntraNode2Full = list(map(lambda x: g.id_dict.get(x), gI.id_dict))
pr_g_on_I = pr_g[idx_IntraNode2Full]

In [ ]:
g.in_strength()
g.out_strength()
gI._in_strength = g._in_strength[idx_IntraNode2Full]
gI._out_strength = g._out_strength[idx_IntraNode2Full]

array([8.16396155e+05, 7.08514057e+03, 5.71352999e+06, ...,
       1.08117884e+02, 4.10402067e+06, 2.87302290e+02], shape=(119612,))

array([3.81018551e+04, 7.44939032e-01, 1.46561689e+06, ...,
       0.00000000e+00, 1.04912717e+06, 6.36961870e+03], shape=(119612,))

NameError: name 'idx_IntraNode2Full' is not defined

Create the `vU, eU` and `gU`

In [14]:
# find idx of edges containing v and filter edges
idx_at_least_one_ = lambda vI: e['src'].isin(vI['id']) | e['dst'].isin(vI['id'])

# containing vI, vR
idx_eU = idx_at_least_one_(vI)

# select the edge ING
eU = edge_idx(idx_eU)

# reselect vU, since using v leads to inactive nodes not well-handled by the DiGraph
vU = unique_nodes_from(eU.iloc[:,:2])

In [15]:
kwargs_graph.update({'perc_intra_nodes' : perc_intra_nodes, 'seed' : seed, 'graph_kind' : "intra_inter",})
gU = sp.graphs.DiGraph(vU, eU, **kwargs_graph)
# g.load_or_create_degrees()

Calculate the Page-Range for `gU`

In [16]:
pr_gU = pagerank_power(gU.adj, p=0.85, max_iter=100,
                   tol=1e-06, personalize=None, reverse=True)

In [17]:
# filtering using the strategy above
idx_UnitedNode2Full = list(map(lambda x: g.id_dict.get(x), gU.id_dict))
pr_g_on_U = pr_g[idx_UnitedNode2Full]

In [18]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([pr_g_on_I.min(), pr_g_on_I.max()],
        [pr_g_on_I.min(), pr_g_on_I.max()],
        'r--')

ms, alpha = 30, 0.6
ax.scatter(pr_g_on_I, pr_gI, alpha=alpha, label = f"Intra PR", marker = 'o', c = dep.obs_color, s = ms)
ax.scatter(pr_g_on_U, pr_gU, alpha=alpha, label = f"United PR", marker = 'x', c = dep.ref_model_color, s = ms)
ax.set(
    xlabel='Full-PR on Intra/United',
    ylabel='Intra/United',
    title='PageRank: Intra/United VS Full',
    xscale='log',
    yscale='log'
)
leg = ax.legend()
for lh in leg.legend_handles:
    lh.set_alpha(1)
    lh.set_sizes([60])
ax.grid(False)
utils.save_fig(fig, full_path = g.plots_general_dir + "/PageRanks_Full.pdf")
plt.close()

[Text(0.5, 0, 'Full-PR on Intra/United'),
 Text(0, 0.5, 'Intra/United'),
 Text(0.5, 1.0, 'PageRank: Intra/United VS Full'),
 None,
 None]

Compare the United PageRank vs Full PageRank over all the `vU` 

In [19]:
idx_IntraNode2United = list(map(lambda x: gU.id_dict.get(x), gI.id_dict))
pr_gU_on_Intra = pr_gU[idx_IntraNode2United]

In [20]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([pr_g_on_I.min(), pr_g_on_I.max()],
        [pr_g_on_I.min(), pr_g_on_I.max()],
        'r--')

ms, alpha = 30, 0.6
ax.scatter(pr_g_on_I, pr_gI, alpha=alpha, label = f"Intra PR", marker = 'o', c = dep.obs_color, s = ms)
ax.scatter(pr_g_on_I, pr_gU_on_Intra, alpha=alpha, label = f"United PR", marker = 'x', c = dep.ref_model_color, s = ms)
ax.set(
    xlabel='Full-PR on Intra',
    ylabel='Intra/United',
    title='PageRank: Intra/United VS Full',
    xscale='log',
    yscale='log'
)
leg = ax.legend()
for lh in leg.legend_handles:
    lh.set_alpha(1)
    lh.set_sizes([60])
ax.grid(False)

utils.save_fig(fig, full_path = g.plots_general_dir + "/PageRanks_on_Intra.pdf")
plt.close()

[Text(0.5, 0, 'Full-PR on Intra'),
 Text(0, 0.5, 'Intra/United'),
 Text(0.5, 1.0, 'PageRank: Intra/United VS Full'),
 None,
 None]

Assuming that the Page-Rank was worse than in the synthetic network, now we fit the models to reconstruct the missing part

In [21]:
# Create graph object for lowest level classification
import sparse as ge

kwargs_graph.update({'name' : 'ING', 'perc_intra_nodes' : perc_intra_nodes, 
                               'seed' : seed, 'level' : 0,
                               'full_intra_row' : "intra",
                               })
gI = ge.graphs.DiGraph(vI, eI, **kwargs_graph)
load_or_create_degrees(gI)

ModuleNotFoundError: No module named 'sparse'

In [ ]:
# create another dictionary for gI
kwargs_model = kwargs_graph.copy()
kwargs_model.update({"name" : "Invariant", "fit_method" : "fit_intra"})
model = ge.ScaleInvariantModel(gI, **kwargs_model)

In [ ]:
# load the invariant model on the gI or fit it
path_param = model.vars_dir + "/param.csv"
if os.path.exists(path_param):
    from graph_ensembles.utils import load_array
    print(f'-Load',)
    # load the param
    model.param = np.expand_dims(load_array(path_param), axis = 0) # The code needs np.array([#])
    
else: 
    print('-Fit',)
    x0 = [1.8996372e-17]
    model.fit(x0 = x0[0], maxiter = 10, verbose = 2) # self.param inside the fit() function
    # save the param
    np.savetxt(path_param, model.param, delimiter = ",")

# calculate the degrees
load_or_create_degrees(model)

NameError: name 'model' is not defined

In [36]:
import plots.plotting_functions as ppf

In [ ]:
ppf.plots_degree(gI, model, model)

In [ ]:
np.max(np.abs(model._in_degree.sum() - gI._num_edges))
np.max(np.abs(model._out_degree.sum() - gI._num_edges))

1.862645149230957e-09

1.3969838619232178e-09

### Miscellanea